# Ilera-ATLAS · 07_demo: live demo with a free Gradio share link

Free alternative to a Hugging Face Space (Gradio Spaces on free CPU now need a PRO subscription).
Runs the fine-tuned Ilera-ATLAS (N-ATLaS + LoRA) on a **Kaggle T4** (or Colab GPU) and opens the same
demo page as `serve/space/app.py`, with the **red-flag safety layer**, behind a public
`https://xxxx.gradio.live` link.

* The link is printed below **and saved to the Hugging Face dataset** as `demo/share_url.txt`.
* It stays up while this notebook runs: `HOURS` (Kaggle sessions end after 12 h; share links after 72 h).
  Re-run for a new link (user test day, video recording).
* Questions and answers are logged **anonymously** to `ilera-data/logs/` every 5 minutes (`LOG = True`)
  for the user-test evidence. Tell testers not to type names or personal details.
* Optional password: set `DEMO_PASSWORD` (user name `ilera`).

**Kaggle:** GPU T4 x2 · Internet On · secret `HF_TOKEN`, then **Save Version → Save & Run All (Commit)**
so it keeps running with your laptop closed.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"           # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
MODEL = f"{HF_USER}/ilera-atlas-lora-runB"   # the chosen fine-tuned adapter
HOURS = 8                               # how long to keep the demo up
LOG = True                              # anonymous interaction log -> ilera-data/logs/
DEMO_PASSWORD = ""                      # e.g. "naic2026" -> login ilera / naic2026; "" = open link
MAX_NEW = 450
REDFLAGS_URL = ("https://raw.githubusercontent.com/OkeyAmy/Ilera/ilera-atlas-pipeline/serve/space/redflags.csv")

In [ ]:
import os, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "gradio>=5"], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    import glob                                     # API runs: token in a private Kaggle dataset
    for p in glob.glob("/kaggle/input/**/hf_token.txt", recursive=True):
        return open(p).read().strip()
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
assert HF_TOKEN, "No HF_TOKEN found"
os.environ["HF_TOKEN"] = HF_TOKEN

## 1. Answer format, red flags and safety layer (same rules as the Space)

In [ ]:
import csv, io, re, urllib.request
from huggingface_hub import hf_hub_download

SYSTEM = open(hf_hub_download(DATA_REPO, "config/system_prompt.txt", repo_type="dataset", token=HF_TOKEN),
              encoding="utf-8").read().strip()
NOTICE = "\n\n_Decision support for trained health workers. Not a diagnosis._"
ACTION_RE = re.compile(r"ACTION:\s*(TREAT|REFER|URGENT)", re.I)

def load_flags():
    txt = urllib.request.urlopen(REDFLAGS_URL, timeout=60).read().decode("utf-8")
    flags = [v.strip().lower() for row in csv.DictReader(io.StringIO(txt)) for v in row.values() if v and v.strip()]
    return sorted(set(flags), key=len, reverse=True)

FLAGS = load_flags()

def red_flag(question):
    q = " " + re.sub(r"\s+", " ", question.lower()) + " "
    return next((f for f in FLAGS if f in q), None)

def apply_safety(question, answer):
    hit = red_flag(question)
    if hit:
        answer = ACTION_RE.sub("ACTION: URGENT", answer, count=1) if ACTION_RE.search(answer) else "ACTION: URGENT\n" + answer
        answer = f"⚠ Danger sign detected ({hit}): refer to a health facility now.\n\n" + answer
    return answer + NOTICE

_s = apply_safety("baby has convulsions", "ACTION: TREAT\nWHAT TO DO:\n1. x")
assert _s.startswith("⚠") and "ACTION: URGENT" in _s
print(len(FLAGS), "red-flag phrases loaded; safety self-check passed")

## 2. Load Ilera-ATLAS on the GPU

In [ ]:
import gc, torch
from unsloth import FastLanguageModel
globals().pop("model", None); gc.collect(); torch.cuda.empty_cache()
model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=2048, load_in_4bit=True, token=HF_TOKEN)
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None

def generate(question):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, repetition_penalty=1.1,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(apply_safety("A 2-year-old has fever and fast breathing.", generate("A 2-year-old has fever and fast breathing. What should I do?")))

## 3. Anonymous interaction log (user test evidence)

In [ ]:
import datetime, json, uuid
from pathlib import Path
scheduler = None
LOG_DIR = Path("demo_logs"); LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"interactions-{datetime.date.today()}-{uuid.uuid4().hex[:6]}.jsonl"
if LOG:
    from huggingface_hub import CommitScheduler
    scheduler = CommitScheduler(repo_id=DATA_REPO, repo_type="dataset", folder_path=LOG_DIR, path_in_repo="logs",
                                every=5, token=HF_TOKEN)

def log(question, answer):
    if scheduler is None:
        return
    with scheduler.lock, open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"time": datetime.datetime.utcnow().isoformat(timespec="seconds"),
                            "question": question, "answer": answer, "backend": "kaggle-gpu"}, ensure_ascii=False) + "\n")
print("logging to", f"{DATA_REPO}/logs/{LOG_FILE.name}" if LOG else "(off)")

## 4. Start the demo and publish the share link

In [ ]:
import time, gradio as gr
from huggingface_hub import HfApi

def answer(message, history=None):
    q = (message or "").strip()
    if not q:
        return "Please type a question."
    final = apply_safety(q, generate(q))
    log(q, final)
    return final

DESCRIPTION = ("Primary-health-care decision support for **trained Nigerian community health workers**, following "
               "the **2024 National Standing Orders**. Ask in **English, Hausa or Yoruba**. N-ATLaS fine-tuned · "
               "Powered by Awarri.\n\nNot a diagnosis tool. Do not type patient names or personal details. "
               "Questions are logged anonymously to improve the system.")
demo = gr.ChatInterface(answer, title="Ilera-ATLAS", description=DESCRIPTION, cache_examples=False,
                        examples=["A 2-year-old has had fever for 3 days and is breathing fast. What should I do?",
                                  "A pregnant woman has a severe headache and blurred vision.",
                                  "Yaro mai shekara biyu yana da zazzabi da saurin numfashi. Me zan yi?",
                                  "Ọmọ ọdún kan ní ìgbẹ́ gbuuru fún ọjọ́ mẹ́ta. Kí ni kí n ṣe?"])
demo.queue(default_concurrency_limit=1)
_, local_url, share_url = demo.launch(share=True, prevent_thread_lock=True,
                                      auth=("ilera", DEMO_PASSWORD) if DEMO_PASSWORD else None)
print("\n" + "=" * 70 + f"\n  ILERA-ATLAS DEMO: {share_url}\n" + "=" * 70)
until = datetime.datetime.utcnow() + datetime.timedelta(hours=HOURS)
note = f"{share_url}\nstarted {datetime.datetime.utcnow():%Y-%m-%d %H:%M} UTC, up until about {until:%Y-%m-%d %H:%M} UTC\n"
open("share_url.txt", "w").write(note)
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="share_url.txt", path_in_repo="demo/share_url.txt",
                                  repo_id=DATA_REPO, repo_type="dataset", commit_message="demo share link")
print("saved to", f"{DATA_REPO}/demo/share_url.txt")

## 5. Keep the demo running

In [ ]:
end = time.time() + HOURS * 3600
while time.time() < end:
    time.sleep(600)
    print(time.strftime("%H:%M"), "demo up:", share_url, flush=True)
if scheduler is not None:
    scheduler.trigger().result()        # final log upload
demo.close()
print("demo stopped")